# AFRICA GIANTS — Continuous Training on Kaggle

Fine-tunes **McGill-NLP/AfriqueLlama-8B** (Llama 3.1 8B, 20 African languages incl. Swahili)
on Tanzanian business/regulatory data via Unsloth QLoRA.

**Requires T4 or better (sm_70+).** Select *T4 GPU* in Kaggle → Settings → Accelerator.

In [ ]:
# ── GPU compatibility check — fail fast if wrong GPU assigned ─────────────
import os
import torch

# Better CUDA error messages (shows exact op that failed, not just 'device-side assert')
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_USE_CUDA_DSA"]   = "1"

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Enable GPU in Kaggle → Settings → Accelerator → T4 GPU."
    )

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
compute = major * 10 + minor
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"GPU  : {gpu_name}")
print(f"VRAM : {vram_gb:.1f} GB")
print(f"SM   : sm_{compute} (need sm_70+)")

if compute < 70:
    raise RuntimeError(
        f"\n"
        f"  GPU '{gpu_name}' has compute capability sm_{compute}.\n"
        f"  PyTorch + Unsloth require sm_70+ (T4=sm_75, V100=sm_70, A100=sm_80).\n"
        f"  Fix: Kaggle → Settings → Accelerator → select 'T4 GPU' → Save → Run All."
    )

print(f"\nGPU compatibility check PASSED ✓  ({gpu_name}, sm_{compute})")

In [ ]:
# Unsloth prebuilt wheel — no CUDA compilation, no OOM during install
!pip install -q unsloth
!pip install -q datasets huggingface_hub

In [ ]:
import os
import torch
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from huggingface_hub import HfApi, create_repo, login, whoami

print(f"CUDA : {torch.cuda.get_device_name(0)}")
print(f"BF16 : {is_bfloat16_supported()}")

In [ ]:
# HF login — Kaggle secret label must be exactly: AFRICA_GIANTS
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("AFRICA_GIANTS")
login(token=hf_token)
print(f"Logged in as: {whoami(token=hf_token)['name']}")

In [ ]:
# ── Hardcoded repo references ──────────────────────────────────────────────
BASE_MODEL        = "McGill-NLP/AfriqueLlama-8B"   # Llama 3.1 8B, 20 African langs
ADAPTER_REPO      = "prospaprospa007/africa-giants-adapter-v1"
MERGED_MODEL_REPO = "prospaprospa007/africa-giants-model-v1"
DATASET_REPO      = "prospaprospa007/africa-giants-dataset"

SMOKE_TEST     = True   # True = 10 steps; False = full 3-epoch run
MAX_SEQ_LENGTH = 512 if SMOKE_TEST else 2048
LOSS_THRESHOLD = 2.5

api = HfApi(token=hf_token)
for repo_id, repo_type in [
    (ADAPTER_REPO, "model"), (MERGED_MODEL_REPO, "model"), (DATASET_REPO, "dataset")
]:
    create_repo(repo_id=repo_id, repo_type=repo_type, private=True, exist_ok=True, token=hf_token)
    print(f"Ready: {repo_type} {repo_id}")

In [ ]:
# ── Load AfriqueLlama-8B via Unsloth (4-bit QLoRA) ────────────────────────
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
    token=hf_token,
)
tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")
print(f"Loaded: {BASE_MODEL}  ({model.num_parameters()/1e9:.2f}B params)")

In [ ]:
# ── LoRA adapters ─────────────────────────────────────────────────────────
model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
model.print_trainable_parameters()

In [ ]:
# ── Load & format dataset ─────────────────────────────────────────────────
raw_dataset = load_dataset(DATASET_REPO, token=hf_token)
print(raw_dataset)

SYSTEM_PROMPT = (
    "Wewe ni msaidizi wa AI wa biashara za Tanzania. "
    "Unajibu maswali kuhusu sheria za biashara, kodi, usajili wa kampuni, "
    "na kanuni za kifedha kwa Kiswahili na Kiingereza. "
    "You are a Tanzanian business AI assistant answering questions about "
    "regulations, tax, company registration, and financial rules in Swahili and English."
)

def format_example(ex):
    inst = ex.get("instruction", "")
    ctx  = ex.get("input", "") or ""
    out  = ex.get("output", "")
    user = f"Context: {ctx}\n\n{inst}" if ctx.strip() else inst
    msgs = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": user},
        {"role": "assistant", "content": out},
    ]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)}

train_ds = raw_dataset["train"].map(format_example, batched=False)
val_src  = raw_dataset.get("validation") or raw_dataset["train"].select(range(min(10, len(raw_dataset["train"]))))
eval_ds  = val_src.map(format_example, batched=False)
print(f"Train: {len(train_ds)}  Eval: {len(eval_ds)}")

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=train_ds, eval_dataset=eval_ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2, packing=False,
    args=TrainingArguments(
        output_dir="./outputs",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=2 if SMOKE_TEST else 4,
        warmup_steps=2,
        max_steps=10 if SMOKE_TEST else -1,
        num_train_epochs=1 if SMOKE_TEST else 3,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        report_to="none",
        save_strategy="no",
        evaluation_strategy="steps",
        eval_steps=5,
        dataloader_pin_memory=False,
    ),
)
print("Starting training...")
stats = trainer.train()
print(f"Done. Runtime: {stats.metrics['train_runtime']:.1f}s")

In [ ]:
# ── Validation loss gate ──────────────────────────────────────────────────
eval_results    = trainer.evaluate()
validation_loss = eval_results.get("eval_loss", 999.0)
gate_passed     = validation_loss <= LOSS_THRESHOLD
print(f"Val loss: {validation_loss:.4f}  threshold: {LOSS_THRESHOLD}  → {'PASSED ✓' if gate_passed else 'FAILED ✗'}")

In [ ]:
# ── Push LoRA adapter + model card ────────────────────────────────────────
if gate_passed:
    print(f"Pushing adapter to {ADAPTER_REPO}...")
    model.push_to_hub_merged(ADAPTER_REPO, tokenizer, save_method="lora", token=hf_token)

    card = f"""---
language:
- sw
- en
license: llama3.1
base_model: {BASE_MODEL}
tags:
- llama-3.1
- african-languages
- swahili
- tanzanian-business
- qlora
- unsloth
- peft
- lora
pipeline_tag: text-generation
---

# Africa Giants — Tanzanian Business AI (LoRA Adapter)

QLoRA fine-tune of [McGill-NLP/AfriqueLlama-8B](https://huggingface.co/McGill-NLP/AfriqueLlama-8B)
on Tanzanian business, tax, company registration, and financial regulation data.

**Base model:** Llama 3.1 8B pre-trained on 20 African languages including Swahili.  
**Languages:** Swahili (sw), English (en)  
**Training:** QLoRA r=16 via Unsloth on Kaggle T4  
**Validation loss:** {validation_loss:.4f}

## Usage
```python
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    "{ADAPTER_REPO}", max_seq_length=2048, load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
```
"""
    api.upload_file(
        path_or_fileobj=card.encode(), path_in_repo="README.md",
        repo_id=ADAPTER_REPO, repo_type="model", token=hf_token,
    )
    print(f"Adapter + model card pushed to {ADAPTER_REPO} ✓")
else:
    print(f"Gate failed — not pushing (loss {validation_loss:.4f} > {LOSS_THRESHOLD})")

In [ ]:
# ── Optional: merge & push full 16-bit model (off by default) ─────────────
MERGE_AND_PUSH = False
if MERGE_AND_PUSH and gate_passed:
    model.push_to_hub_merged(MERGED_MODEL_REPO, tokenizer, save_method="merged_16bit", token=hf_token)
    print(f"Merged model → {MERGED_MODEL_REPO} ✓")
else:
    print("Skipping merged model push.")